In [ ]:
instructions = """You're a course teaching assistant.You're given a question from a course student and your task is to answer it.If you want to look up information, use the search function.Use as many keywords from the user question as possible when making first requests.Make multiple searches.Try to expand your search by using new keywordsbased on the results you get from the search.At the end, ask if there are other areas that the user wants to explore.""".strip()

In [ ]:
from dotenv import load_dotenvimport osimport jsonload_dotenv()from ingest import load_lessons_data, build_indexfrom rag_helper import RAGBasefrom openai import OpenAIfrom gitsource import chunk_documents

In [ ]:
documents = load_lessons_data()index = build_index(documents)openai_client = OpenAI(    api_key=os.getenv("GEMINI_API_KEY"),    base_url="https://generativelanguage.googleapis.com/v1beta/openai/")rag = RAGBase(index, openai_client)def search(query, num_results=5):    return rag.search(query, num_results=num_results)

In [ ]:
search_tool = {    "type": "function",    "name": "search",    "description": "Search the FAQ database for entries matching the given query.",    "parameters": {        "type": "object",        "properties": {            "query": {                "type": "string",                "description": "Search query text to look up in the course FAQ."            },            "num_results": {                "type": "integer",                "description": "Number of search results to return.",                "default": 5            }        },        "required": ["query"],        "additionalProperties": False    }}

In [ ]:
def make_call(call):    args = json.loads(call.arguments)    if call.name == "search":        result = search(**args)    else:        result = {"error": "unknown function"}    result_json = json.dumps(result, indent=2)    return {        "type": "function_call_output",        "call_id": call.call_id,        "output": result_json,    }

In [ ]:
question = "I just discovered the course. Can I join it?"messages = [    {"role": "developer", "content": instructions},    {"role": "user", "content": question},]response = openai_client.chat.completions.create(    model="gemini-3.1-flash-lite",    messages=messages,    tools=[search_tool],)messages.extend(response.choices[0].message)has_function_calls = Falsefor item in response.choices[0].message:    if item.type == "function_call":        print("function_call:", item.name, item.arguments)        call_output = make_call(item)        messages.append(call_output)        has_function_calls = True    elif item.type == "message":        print("ASSISTANT:")        print(item.content[0].text)